In [0]:
%python
employee_data = (
    (101, "Alice", 10, 70000),
    (102, "Bob", 20, 85000),
    (103, "Charlie", 10, 65000),
    (104, "Diana", 30, 90000),
    (105, "Evan", 40, 60000),
    (106, "Frank", 20, 75000),
    (107, "Grace", 50, 72000),
    (108, "Henry", 30, 68000),
    (109, "Ivy", 40, 95000),
    (110, "Jack", 50, 80000)
)
employee_column = ["emp_id", "emp_name", "dept_id", "salary"]

department_data = (
    (10, "HR"),
    (20, "Engineering"),
    (30, "Finance"),
    (40, "Marketing"),
    (50, "Operations")
)
department_column = ["dept_id", "dept_name"]

df_employee = spark.createDataFrame(employee_data, employee_column)
df_department = spark.createDataFrame(department_data, department_column)

display(df_employee)
display(df_department)

df_employee.createOrReplaceTempView("emp_temp_view")
df_department.createOrReplaceTempView("dept_temp_view")


**highest salary per department name**

In [0]:
%python
from pyspark.sql.functions import max

# join to get the department name and salary
df_join = (df_employee
           .join(df_department, df_employee.dept_id == df_department.dept_id, "left")
           .select(df_department.dept_name, df_employee.salary)
           .orderBy(df_department.dept_name)
)
display(df_join)

# group by with department name
df_grp_by = (df_join
             .groupBy(df_join.dept_name)
             .agg(max(df_join.salary)).alias("max_salary")

)
display(df_grp_by)

In [0]:
with cte as
(
  select
    d.dept_name,
    e.salary
  from emp_temp_view as e
  inner join dept_temp_view as d
  on e.dept_id = d.dept_id
)
select
  dept_name,
  max(salary) as max_salary
from cte
group by dept_name


In [0]:
with cte as
(
  select
    d.dept_name,
    e.salary,
    row_number() over(partition by d.dept_name order by e.salary desc) as rn
  from emp_temp_view as e
  inner join dept_temp_view as d
  on e.dept_id = d.dept_id
)
select
  dept_name,
  salary
from cte
where rn = 1


### **get the employee name for each department who is getting highest salary**

In [0]:
%python
from pyspark.sql.functions import desc, row_number, col
from pyspark.sql.window import Window

# join to get the department name
df_join = (df_employee
          .join(df_department, df_employee.dept_id == df_department.dept_id, "inner")
          .select(df_department.dept_name, df_employee.emp_name, df_employee.salary)
          .orderBy(df_department.dept_name)
)
display(df_join)

# set the window spec
window_spec = Window.partitionBy(df_join.dept_name).orderBy(desc(df_join.salary))

# get the max salary using row_number
df_rn = (df_join
               .withColumn("rn", row_number().over(window_spec))

)
display(df_rn)

# filter with max salary
df_max_salary = (df_rn
                 .filter(df_rn.rn == 1)
                 .select(df_rn.dept_name, df_rn.emp_name, df_rn.salary)
)
display(df_max_salary)

# with single step
window_spec_single = Window.partitionBy(df_department.dept_name).orderBy(desc(df_employee.salary))

display(df_employee
        .join(df_department, df_employee.dept_id == df_department.dept_id, "inner")
        .select(df_employee.emp_name, df_employee.emp_name, df_employee.salary, df_department.dept_name)
        .withColumn("rn", row_number().over(window_spec_single))
        .filter(col("rn") == 1)
        .select(df_department.dept_name, df_employee.emp_name, df_employee.salary)
)

In [0]:
with cte as
(
  select
    d.dept_name,
    e.emp_name,
    e.salary,
    row_number() over(partition by d.dept_name order by e.salary desc) as rn
  from emp_temp_view as e
  inner join dept_temp_view as d
  on e.dept_id = d.dept_id
)
select
  dept_name,
  emp_name,
  salary
from cte
where rn = 1